# PicoRV32 + CIM SoC — PYNQ 端 (无 pyserial, 配合 PC 串口读取)

**运行位置**: PYNQ-Z2 Jupyter Notebook  
**假设**: pyserial **不可用** (PYNQ 默认环境可能没有 pyserial)

## 工作模式

PYNQ 和 PC **同时**运行各自的 notebook:

| 端 | Notebook | 职责 |
|----|----------|------|
| PYNQ (本文件) | `picorv32_mlp_pynq_no_pyserial.ipynb` | 加载 bitstream, golden model 推理, 导出结果 |
| PC | `picorv32_mlp_pc_no_pyserial.ipynb` | 读取 UART, 解析推理结果, 对比 golden |

## 硬件连接

USB-TTL 适配器的 USB 端插入 **PC** (不是 PYNQ):

```
PYNQ-Z2 PMODA              USB-TTL                PC
  Pin 1 (Y18, TX) -------> RXD                     |
  Pin 5 (GND)     -------> GND    ----USB---->  /dev/ttyUSB0
```

## 上传文件清单

| 文件 | 说明 |
|------|------|
| `cim_rv32_soc.bit` | PicoRV32 版 bitstream (checkpoint2) |
| `small_mlp_data/` | 模型数据目录 (权重/偏置/测试图片/量化参数) |
| `cim_soc.bit` + `cim_soc.hwh` | (可选) PS 版 bitstream 用于交叉验证 |

## 1. 加载 Bitstream (Pure-PL)

加载后 PicoRV32 立即开始执行固件，UART 输出会发送到 PMODA Pin 1。

**重要**: 先在 PC 上启动 UART 读取，再运行此 Cell (或之后按 BTN0 复位)。

In [ ]:
import subprocess, os, time, glob, struct
import numpy as np

BIT_FILE = "cim_rv32_soc.bit"

assert os.path.exists(BIT_FILE), f"{BIT_FILE} not found! Please upload."

print(f"Loading {BIT_FILE} ({os.path.getsize(BIT_FILE)/1024:.0f} KB)...")
print(">>> Make sure PC-side UART reader is running FIRST <<<")
print()

try:
    subprocess.run(
        ["sudo", "bash", "-c", f"cat {BIT_FILE} > /dev/xdevcfg"],
        check=True, timeout=30
    )
    print("Bitstream loaded!")
except Exception as e:
    print(f"Method 1 failed: {e}")
    try:
        subprocess.run(["fpgautil", "-b", BIT_FILE], check=True, timeout=30)
        print("Bitstream loaded! (fpgautil)")
    except Exception as e2:
        print(f"Method 2 failed: {e2}")
        print("Manual: sudo bash -c 'cat cim_rv32_soc.bit > /dev/xdevcfg'")

print()
print("PicoRV32 is now executing firmware!")
print("  LED0 (R14): CIM done indicator")
print("  LED1 (P14): heartbeat (~1Hz blink = CPU running)")
print()
print("UART output is being sent to PMODA Pin 1 (Y18) --> PC")
print("If PC missed the output, press BTN0 to restart PicoRV32")

## 2. Golden Model 推理 (small_mlp_data)

在 PYNQ ARM PS 上加载模型数据，做 bit-accurate INT8 推理。
结果将保存为文件，下载到 PC 供对比。

In [ ]:
DATA_DIR = "small_mlp_data"

assert os.path.isdir(DATA_DIR), (
    f"{DATA_DIR}/ not found! Please upload.\n"
    "Generated by prepare_picorv32_env.ipynb on host PC."
)

# ---- Helper functions ----
def read_hex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

def unpack_weights(chunk_path, out_dim, in_dim):
    chunks = read_hex_u32(chunk_path)
    w = np.zeros((out_dim, in_dim), dtype=np.int8)
    idx = 0
    for ob in range((out_dim + 15) // 16):
        for ib in range((in_dim + 15) // 16):
            for ch in range(64):
                r, cg = ch // 4, ch % 4
                word = chunks[idx]; idx += 1
                for b in range(4):
                    oi = ob * 16 + r
                    ii = ib * 16 + cg * 4 + b
                    if oi < out_dim and ii < in_dim:
                        val = (word >> (b * 8)) & 0xFF
                        w[oi, ii] = np.int8(struct.unpack('b', bytes([val]))[0])
    return w

def unpack_bias(path):
    return np.array([np.int32(np.uint32(v)) for v in read_hex_u32(path)], dtype=np.int32)

def hw_mvm(x_u8, w_i8, b_i32, zp, mult, shift, relu):
    """Bit-accurate INT8 MVM matching CIM hardware."""
    x_eff = np.clip(x_u8.astype(np.int32) - zp, 0, 511)
    acc = w_i8.astype(np.int32) @ x_eff + b_i32
    if relu:
        acc = np.maximum(acc, 0)
    out = np.zeros(len(acc), dtype=np.int32)
    for i in range(len(acc)):
        scaled = (acc[i] * mult) >> shift
        out[i] = max(-128, min(127, scaled))
    return out.astype(np.int8)

# ---- Load model ----
qp = read_hex_u32(f"{DATA_DIR}/quant_params.hex")
zps = read_hex_u32(f"{DATA_DIR}/zero_points.hex")
fc1_mult, fc1_shift = qp[0], qp[1]
fc2_mult, fc2_shift = qp[2], qp[3]
hw_zp1 = np.int32(np.uint32(zps[0]))
hw_zp2 = np.int32(np.uint32(zps[1]))

w1 = unpack_weights(f"{DATA_DIR}/fc1_weight_tiles.hex", 16, 784)
b1 = unpack_bias(f"{DATA_DIR}/fc1_bias.hex")
w2 = unpack_weights(f"{DATA_DIR}/fc2_weight_tiles.hex", 10, 16)
b2 = unpack_bias(f"{DATA_DIR}/fc2_bias.hex")

print(f"Model loaded from {DATA_DIR}/")
print(f"  FC1: {w1.shape}, FC2: {w2.shape}")
print(f"  Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})")
print(f"  ZP: fc1={hw_zp1}, fc2={hw_zp2}")

In [ ]:
# ---- Run golden inference on all test images ----
img_dir = f"{DATA_DIR}/test_images"
img_files = sorted(glob.glob(f"{img_dir}/img_*.hex"))
# Filter out label files
img_files = [f for f in img_files if "_label" not in f]
n_images = len(img_files)
print(f"Test images: {n_images}")
print()

golden_results = []
for img_path in img_files:
    base = os.path.basename(img_path).replace(".hex", "")
    label_path = os.path.join(img_dir, f"{base}_label.txt")
    img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
    label = int(open(label_path).read().strip())

    # FC1: 784->16, ReLU
    fc1_out = hw_mvm(img_u8, w1, b1[:16], hw_zp1, fc1_mult, fc1_shift, relu=True)
    fc1_u8 = fc1_out.astype(np.uint8)
    # FC2: 16->10, no activation
    fc2_out = hw_mvm(fc1_u8, w2, b2[:10], hw_zp2, fc2_mult, fc2_shift, relu=False)

    pred = int(np.argmax(fc2_out))
    logits = fc2_out.tolist()
    golden_results.append({
        "idx": base, "label": label, "pred": pred,
        "logits": logits, "correct": pred == label
    })

g_correct = sum(1 for r in golden_results if r["correct"])
print(f"Golden Model Accuracy: {g_correct}/{n_images} ({100*g_correct/n_images:.0f}%)")
print()
print(f"{'Image':>12} {'Label':>6} {'Pred':>5} {'Result':>8} {'Logits'}")
print("-" * 65)
for r in golden_results:
    status = "OK" if r["correct"] else "WRONG"
    logits_str = str(r['logits'])
    print(f"{r['idx']:>12} {r['label']:6d} {r['pred']:5d} {status:>8} {logits_str}")

## 3. 导出 Golden Results

保存 golden 结果为文本文件，下载到 PC 供 `picorv32_mlp_pc_no_pyserial.ipynb` 对比。

In [ ]:
GOLDEN_OUT = "pynq_golden_results.txt"

with open(GOLDEN_OUT, "w") as f:
    f.write("# PYNQ Golden Model Results (bit-accurate INT8)\n")
    f.write(f"# Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"# Model: small_mlp (784->16->10)\n")
    f.write(f"# Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})\n")
    f.write(f"# ZP: fc1={hw_zp1}, fc2={hw_zp2}\n")
    f.write(f"# Format: idx label pred correct logits...\n")
    for i, r in enumerate(golden_results):
        logits_str = " ".join(str(x) for x in r["logits"])
        ok = 1 if r["correct"] else 0
        f.write(f"{i} {r['label']} {r['pred']} {ok} {logits_str}\n")

print(f"Saved: {GOLDEN_OUT}")
print(f"  {n_images} images, accuracy {g_correct}/{n_images}")
print()
print(">>> Download this file to PC for cross-verification <<<")
print(f"  PC notebook expects: pynq_golden_results.txt")

## 4. [可选] PS 版 CIM 硬件交叉验证

如果有 checkpoint1 的 `cim_soc.bit` + `cim_soc.hwh`，
可以用 PS (ARM + MMIO) 驱动同一个 CIM IP 跑 20 张图，
确认 PS 版和 Golden Model bit-exact 一致。

**注意**: 加载 PS 版 bitstream 会覆盖当前 PicoRV32 bitstream。

In [ ]:
PS_BIT = "cim_soc.bit"
PS_HWH = "cim_soc.hwh"

if not all(os.path.exists(f) for f in [PS_BIT, PS_HWH]):
    print(f"Skip PS cross-validation: need {PS_BIT} + {PS_HWH} (checkpoint1)")
    ps_results = None
else:
    from pynq import Overlay, MMIO

    ol = Overlay(PS_BIT)
    print("PS Overlay loaded!")
    mmio = MMIO(0x40000000, 0x4000)

    # Quick connectivity test
    mmio.write(0x010, 784)
    assert mmio.read(0x010) == 784, "AXI R/W failed!"
    print("AXI connectivity OK")

    # ---- CIM driver functions (same as firmware.c) ----
    def soft_reset():
        mmio.write(0x000, 0x4)

    def configure(in_dim, out_dim, zp, mult, shift, relu):
        mmio.write(0x010, in_dim)
        mmio.write(0x014, out_dim)
        mmio.write(0x018, (in_dim + 15) // 16)
        mmio.write(0x01C, (out_dim + 15) // 16)
        mmio.write(0x020, mult)
        mmio.write(0x024, shift)
        mmio.write(0x028, int(zp) & 0xFFFFFFFF)
        mmio.write(0x02C, 1 if relu else 0)

    def load_weights(chunks):
        mmio.write(0x044, 0)
        mmio.write(0x04C, 0x02)
        for c in chunks:
            mmio.write(0x048, int(c) & 0xFFFFFFFF)
        mmio.write(0x04C, 0x00)

    def load_bias(bias, n):
        for i in range(n):
            mmio.write(0x2000 + 4 * i, int(bias[i]) & 0xFFFFFFFF)

    def load_input(data):
        padded = ((len(data) + 15) // 16) * 16
        for i in range(padded):
            mmio.write(0x1000 + 4 * i, int(data[i]) if i < len(data) else 0)

    def start_wait(t=5.0):
        mmio.write(0x000, 0x2)
        mmio.write(0x000, 0x1)
        t0 = time.time()
        while not (mmio.read(0x004) & 0x2):
            if time.time() - t0 > t:
                raise TimeoutError("CIM compute timeout!")
        return time.time() - t0

    def read_output(n):
        return [np.int8(mmio.read(0x100 + 4 * i) & 0xFF) for i in range(n)]

    def read_pred():
        return mmio.read(0x040)

    # ---- Load weight data as raw chunks ----
    fc1_wc = read_hex_u32(f"{DATA_DIR}/fc1_weight_tiles.hex")
    fc1_bias_raw = read_hex_u32(f"{DATA_DIR}/fc1_bias.hex")
    fc2_wc = read_hex_u32(f"{DATA_DIR}/fc2_weight_tiles.hex")
    fc2_bias_raw = read_hex_u32(f"{DATA_DIR}/fc2_bias.hex")

    # ---- Run PS inference on all images ----
    ps_results = []
    for img_path in img_files:
        base = os.path.basename(img_path).replace(".hex", "")
        label_path = os.path.join(img_dir, f"{base}_label.txt")
        img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
        label = int(open(label_path).read().strip())

        # FC1
        soft_reset()
        configure(784, 16, hw_zp1, fc1_mult, fc1_shift, relu=True)
        load_weights(fc1_wc)
        load_bias(fc1_bias_raw, 16)
        load_input(img_u8)
        start_wait()
        fc1_out = np.array(read_output(16), dtype=np.uint8)

        # FC2
        configure(16, 10, hw_zp2, fc2_mult, fc2_shift, relu=False)
        load_weights(fc2_wc)
        load_bias(fc2_bias_raw, 10)
        load_input(fc1_out)
        start_wait()
        logits = read_output(10)
        pred = int(read_pred())

        ps_results.append({
            "idx": base, "label": label, "pred": pred,
            "logits": logits, "correct": pred == label
        })
        status = "OK" if pred == label else "WRONG"
        print(f"  {base}: label={label}, pred={pred} {status}")

    ps_correct = sum(1 for r in ps_results if r["correct"])
    print(f"\nPS CIM: {ps_correct}/{n_images} correct")

## 5. 三方对比汇总

Golden Model (Python INT8) vs PS CIM Hardware vs PicoRV32 (UART on PC)

In [ ]:
print("=" * 60)
print(" Summary")
print("=" * 60)
print()

# Golden model
print(f"[Golden Model (Python INT8)]  {g_correct}/{n_images} correct")

# PS CIM
if ps_results is not None:
    ps_c = sum(1 for r in ps_results if r["correct"])
    print(f"[PS CIM Hardware (MMIO)]      {ps_c}/{n_images} correct")

    # Check Golden vs PS bit-exact match
    all_match = True
    for i in range(n_images):
        if golden_results[i]["pred"] != ps_results[i]["pred"]:
            all_match = False
            print(f"  MISMATCH image {i}: golden={golden_results[i]['pred']}, PS={ps_results[i]['pred']}")
    if all_match:
        print("  >>> Golden vs PS: ALL bit-exact match! <<<")
else:
    print("[PS CIM Hardware]             Not run (need checkpoint1 files)")

# PicoRV32
print("[PicoRV32 (UART)]             Check PC-side notebook for result")

# PC-side result file (if uploaded back)
PC_RESULT = "pc_uart_result.txt"
if os.path.exists(PC_RESULT):
    print(f"\n--- PC UART Result (uploaded) ---")
    pc = {}
    with open(PC_RESULT) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split()
            if len(parts) >= 2:
                pc[parts[0]] = parts[1]

    if "predicted" in pc and "expected" in pc:
        hw_pred = int(pc["predicted"])
        hw_label = int(pc["expected"])
        print(f"  HW: pred={hw_pred}, label={hw_label}")

        matched = [r for r in golden_results if r["label"] == hw_label]
        if matched:
            g = matched[0]
            print(f"  Golden: pred={g['pred']}")
            if hw_pred == g["pred"]:
                print("  >>> PicoRV32 vs Golden: MATCH (bit-exact!) <<<")
            else:
                print("  >>> PicoRV32 vs Golden: MISMATCH <<<")
else:
    print(f"\nTo complete cross-check: upload {PC_RESULT} from PC.")

print()
print(f"Golden results file: {GOLDEN_OUT}")
print("Download it and place next to the PC notebook for verification.")

## 6. 复位 PicoRV32 (重新发送 UART)

如果 PC 端漏掉了 UART 输出，需要重新触发。

- **方式 A**: 按 PYNQ 上的 BTN0 (物理复位按钮)
- **方式 B**: 重新加载 bitstream (本 Cell)

In [ ]:
# Reload PicoRV32 bitstream to re-trigger UART output
# Make sure PC-side UART reader is ready before running!

RELOAD = False  # Set to True to reload

if RELOAD:
    print(">>> Confirm PC-side UART reader is running! <<<")
    print("Reloading bitstream in 3 seconds...")
    time.sleep(3)

    try:
        subprocess.run(
            ["sudo", "bash", "-c", f"cat {BIT_FILE} > /dev/xdevcfg"],
            check=True, timeout=30
        )
        print("Bitstream reloaded! PicoRV32 is running.")
        print("UART output should appear on PC now.")
    except Exception as e:
        print(f"Reload failed: {e}")
        print("Press BTN0 on the board instead.")
else:
    print("Set RELOAD = True to reload bitstream.")
    print("Or press BTN0 on the PYNQ board.")

## 总结

本 notebook 在 **PYNQ** 上完成:

1. 加载 PicoRV32 bitstream (UART 输出给 PC)
2. 用 small_mlp_data 做 Golden Model 推理
3. 导出 `pynq_golden_results.txt` 供 PC 对比
4. (可选) PS 版 CIM 交叉验证

```
PYNQ 端 (本 notebook)                    PC 端
  |
  |-- Load bitstream --> PicoRV32 UART --> USB-TTL --> PC reads UART
  |
  |-- Golden inference --> pynq_golden_results.txt
  |                               |
  |                               +--download--> PC compares
  |
  |-- (optional) PS CIM cross-validation
```

**不需要 pyserial**。UART 读取完全在 PC 端完成。